# Sistem Case-Based Reasoning (CBR)
## Analisis Putusan Pidana – Kejahatan Terhadap Ketertiban Umum

**Mata Kuliah:** Penalaran Komputer — SubCPMK-3  
**Sumber Data:** Direktori Putusan Mahkamah Agung RI  
**Jumlah Dokumen:** 31 putusan  
**Model:** TF-IDF+SVM | TF-IDF+Naive Bayes | IndoBERT+SVM

---
## ⚠️ Sebelum Menjalankan
Pastikan sudah menjalankan dari terminal di folder root project:
```bash
python scripts/01_proses_pdf.py
python scripts/02_representasi.py
```
Lalu jalankan: **Kernel → Restart & Run All**

## 0. Setup & Import Library

In [ ]:
# !pip install scikit-learn pandas numpy matplotlib seaborn tqdm sentence-transformers

In [ ]:
import re, json, pickle, warnings, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)
warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# Deteksi ROOT_DIR otomatis
_nb_file = globals().get('__vsc_ipynb_file__', '')
if _nb_file:
    ROOT_DIR = Path(_nb_file).resolve().parent.parent
else:
    _cwd = Path(os.getcwd()).resolve()
    if _cwd.name == 'notebooks':
        ROOT_DIR = _cwd.parent
    elif (_cwd / 'notebooks').exists() and (_cwd / 'scripts').exists():
        ROOT_DIR = _cwd
    else:
        ROOT_DIR = _cwd.parent

RAW_DIR       = ROOT_DIR / 'data' / 'raw'
PROCESSED_DIR = ROOT_DIR / 'data' / 'processed'
EVAL_DIR      = ROOT_DIR / 'data' / 'eval'
RESULTS_DIR   = ROOT_DIR / 'data' / 'results'
FIGS_DIR      = ROOT_DIR / 'figures'
for d in [EVAL_DIR, RESULTS_DIR, FIGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('✓ Setup selesai')
print(f'  Root    : {ROOT_DIR}')
print(f'  Raw     : {RAW_DIR} → exists: {RAW_DIR.exists()}')
print(f'  Process : {PROCESSED_DIR} → exists: {PROCESSED_DIR.exists()}')
if not (PROCESSED_DIR / 'cases.json').exists():
    print('\n⚠️  cases.json tidak ditemukan! Jalankan scripts/02_representasi.py dulu.')
else:
    print('\n✓ cases.json ditemukan → siap lanjut!')

---
## Tahap 1: Membangun Case Base

In [ ]:
txt_files = sorted(RAW_DIR.glob('case_*.txt'))
meta_path = RAW_DIR / 'metadata_raw.json'
print(f'Jumlah file .txt : {len(txt_files)}')
if meta_path.exists():
    with open(meta_path, encoding='utf-8') as f:
        metadata_raw = json.load(f)
    meta_df = pd.DataFrame(metadata_raw)
    print(f'Entri metadata   : {len(meta_df)}')
    print(meta_df['status'].value_counts().to_string())
    display(meta_df[['case_id','file_asli','word_count','status']].head(5))
if txt_files:
    sample = txt_files[0].read_text(encoding='utf-8', errors='ignore')
    print(f'\nContoh teks ({txt_files[0].name}):')
    print('-'*60)
    print(sample[:400])

---
## Tahap 2: Case Representation

In [ ]:
with open(PROCESSED_DIR / 'cases.json', encoding='utf-8') as f:
    cases_data = json.load(f)
df = pd.DataFrame(cases_data)
print(f'Total kasus  : {len(df)}')
print(f'text_full OK : {(df["text_full"].str.len() > 100).sum()}/{len(df)}')
print(f'\nDistribusi label:')
print(df['label'].value_counts().to_string())
display(df[['case_id','no_perkara','tanggal','pengadilan','terdakwa','pasal','vonis','label','word_count']].head(10))
df.drop(columns=['text_full'], errors='ignore').to_csv(PROCESSED_DIR / 'cases.csv', index=False, encoding='utf-8-sig')
print('\n✓ cases.csv disimpan')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
lc = df['label'].value_counts()
axes[0].bar(lc.index, lc.values, color=['#4C72B0','#DD8452','#55A868','#C44E52','#9B59B6'][:len(lc)], alpha=0.85)
axes[0].set_title('Distribusi Label Putusan', fontweight='bold')
axes[0].set_xlabel('Label'); axes[0].set_ylabel('Jumlah')
axes[0].tick_params(axis='x', rotation=20)
for i, v in enumerate(lc.values): axes[0].text(i, v+0.1, str(v), ha='center', fontweight='bold')
axes[1].hist(df['word_count'].dropna(), bins=15, color='#4C72B0', edgecolor='white', alpha=0.85)
axes[1].set_title('Distribusi Jumlah Kata', fontweight='bold')
axes[1].set_xlabel('Jumlah Kata'); axes[1].set_ylabel('Frekuensi')
axes[1].axvline(df['word_count'].median(), color='red', linestyle='--', label=f'Median: {int(df["word_count"].median())}')
axes[1].legend()
pc = df['pengadilan'].value_counts().head(7)
axes[2].barh(pc.index, pc.values, color='#55A868', alpha=0.85)
axes[2].set_title('Top 7 Pengadilan', fontweight='bold')
axes[2].set_xlabel('Jumlah Kasus')
plt.suptitle('Eksplorasi Data – Putusan Ketertiban Umum (31 Dokumen)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'data_exploration.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ figures/data_exploration.png disimpan')

---
## Tahap 3A: Case Retrieval – TF-IDF

In [ ]:
df_labeled = df[(df['label'] != 'tidak_diketahui') & (df['text_full'].str.len() > 100)].copy().reset_index(drop=True)
print(f'Kasus berlabel  : {len(df_labeled)}')
print(f'Tidak diketahui : {len(df) - len(df_labeled)}')
print(df_labeled['label'].value_counts().to_string())
texts    = df_labeled['text_full'].tolist()
labels   = df_labeled['label'].tolist()
ids      = df_labeled['case_id'].tolist()
all_texts = df['text_full'].fillna('').tolist()
all_ids   = df['case_id'].tolist()
le = LabelEncoder()
y  = le.fit_transform(labels)
print(f'\nKelas: {le.classes_}')

In [ ]:
try:
    train_idx, test_idx = train_test_split(range(len(texts)), test_size=0.2, random_state=42, stratify=y)
except ValueError:
    train_idx, test_idx = train_test_split(range(len(texts)), test_size=0.2, random_state=42)
train_idx, test_idx = list(train_idx), list(test_idx)
print(f'Train : {len(train_idx)} kasus')
print(f'Test  : {len(test_idx)} kasus')
with open(EVAL_DIR / 'split_info.json', 'w') as f:
    json.dump({'train_idx': train_idx, 'test_idx': test_idx}, f)
print('✓ split_info.json disimpan')

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True, min_df=1, strip_accents='unicode')
X_all    = tfidf.fit_transform(texts)
X_train  = X_all[train_idx]
X_test   = X_all[test_idx]
y_train  = y[train_idx]
y_test   = y[test_idx]
X_corpus = tfidf.transform(all_texts)
print(f'Vocabulary size : {len(tfidf.vocabulary_):,}')
print(f'Train matrix    : {X_train.shape}')
print(f'Test matrix     : {X_test.shape}')

In [ ]:
def retrieve(query: str, k: int = 5) -> list:
    """
    Retrieve top-k kasus paling mirip (TF-IDF + Cosine Similarity).
    1) Pre-process query
    2) Hitung vektor query via TF-IDF
    3) Hitung cosine-similarity dengan semua case vectors
    4) Kembalikan top-k case_id
    """
    q_vec = tfidf.transform([query])
    sims  = cosine_similarity(q_vec, X_corpus).flatten()
    top_k = np.argsort(sims)[::-1][:k]
    return [(all_ids[i], round(float(sims[i]), 4)) for i in top_k]

q = 'terdakwa bersama melakukan kekerasan di muka umum'
print(f'Test retrieve: "{q}"\nTop-5:')
for rank, (cid, score) in enumerate(retrieve(q), 1):
    row = df[df['case_id']==cid].iloc[0]
    print(f'  {rank}. {cid} | sim={score} | {row["label"]} | {row["no_perkara"]}')

In [ ]:
svm_model = LinearSVC(C=1.0, max_iter=5000, random_state=42)
svm_model.fit(X_train, y_train)
print('✓ TF-IDF + SVM dilatih')
nb_model = MultinomialNB(alpha=0.1)
nb_model.fit(X_train, y_train)
print('✓ TF-IDF + Naive Bayes dilatih')
with open(PROCESSED_DIR / 'models.pkl', 'wb') as f:
    pickle.dump({'tfidf': tfidf, 'svm': svm_model, 'nb': nb_model, 'le': le}, f)
print('✓ Model disimpan: data/processed/models.pkl')

In [ ]:
eval_queries = [{'query_id': f'q_{ids[i]}', 'query_text': texts[i][:300], 'ground_truth': ids[i], 'true_label': labels[i]} for i in test_idx]
with open(EVAL_DIR / 'queries.json', 'w', encoding='utf-8') as f:
    json.dump(eval_queries, f, ensure_ascii=False, indent=2)
print(f'✓ queries.json disimpan ({len(eval_queries)} query)')

---
## Tahap 3B: Case Retrieval – IndoBERT Embedding
> ⏱️ Proses ini **10–20 menit** di CPU. Biarkan berjalan hingga selesai.

In [ ]:
from sentence_transformers import SentenceTransformer
BERT_MODEL = 'firqaaa/indo-sentence-bert-base'
MAX_CHARS  = 1500
print(f'Memuat model: {BERT_MODEL}')
print('(Download ~400MB jika pertama kali)\n')
bert_model = SentenceTransformer(BERT_MODEL)
print('✓ Model IndoBERT dimuat')

In [ ]:
print('Encoding corpus (31 dokumen)...')
corpus_embeddings = bert_model.encode([t[:MAX_CHARS] for t in all_texts], batch_size=4, show_progress_bar=True, normalize_embeddings=True, convert_to_numpy=True)
print('\nEncoding train...')
train_embeddings = bert_model.encode([texts[i][:MAX_CHARS] for i in train_idx], batch_size=4, show_progress_bar=True, normalize_embeddings=True, convert_to_numpy=True)
print('\nEncoding test...')
test_embeddings = bert_model.encode([texts[i][:MAX_CHARS] for i in test_idx], batch_size=4, show_progress_bar=True, normalize_embeddings=True, convert_to_numpy=True)
print(f'\n✓ Embedding selesai!')
print(f'  Corpus : {corpus_embeddings.shape}')
print(f'  Train  : {train_embeddings.shape}')
print(f'  Test   : {test_embeddings.shape}')

In [ ]:
bert_svm = LinearSVC(C=1.0, max_iter=5000, random_state=42)
bert_svm.fit(train_embeddings, y_train)
print('✓ IndoBERT + SVM dilatih')
with open(PROCESSED_DIR / 'bert_state.pkl', 'wb') as f:
    pickle.dump({'corpus_embeddings': corpus_embeddings, 'bert_svm': bert_svm, 'model_name': BERT_MODEL}, f)
print('✓ bert_state.pkl disimpan')

def retrieve_bert(query: str, k: int = 5) -> list:
    """
    Retrieve top-k kasus menggunakan IndoBERT embedding.
    1) Encode query → embedding vector
    2) Cosine similarity dengan corpus embeddings
    3) Kembalikan top-k case_id
    """
    q_emb = bert_model.encode([query[:MAX_CHARS]], normalize_embeddings=True, convert_to_numpy=True)
    sims  = (q_emb @ corpus_embeddings.T).flatten()
    top_k = np.argsort(sims)[::-1][:k]
    return [(all_ids[i], round(float(sims[i]), 4)) for i in top_k]

q = 'terdakwa bersama melakukan kekerasan di muka umum'
print(f'\nTest retrieve IndoBERT: "{q}"\nTop-5:')
for rank, (cid, score) in enumerate(retrieve_bert(q), 1):
    row = df[df['case_id']==cid].iloc[0]
    print(f'  {rank}. {cid} | sim={score} | {row["label"]} | {row["no_perkara"]}')

---
## Tahap 4: Case Solution Reuse

In [ ]:
case_solutions = {row['case_id']: {'amar_putusan': str(row.get('amar_putusan','')), 'vonis': str(row.get('vonis','')), 'label': str(row.get('label','')), 'no_perkara': str(row.get('no_perkara','')), 'pasal': str(row.get('pasal',''))} for _, row in df.iterrows()}
print(f'✓ Solusi dimuat untuk {len(case_solutions)} kasus')

In [ ]:
def predict_outcome(query: str, model: str = 'svm', k: int = 5) -> dict:
    top_k = retrieve_bert(query, k) if model == 'bert' else retrieve(query, k)
    weights, amars = {}, {}
    for cid, score in top_k:
        sol = case_solutions.get(cid, {})
        lbl = sol.get('label', 'tidak_diketahui')
        weights[lbl] = weights.get(lbl, 0) + score
        if lbl not in amars: amars[lbl] = sol.get('amar_putusan', '')[:200]
    best_label = max(weights, key=weights.__getitem__)
    if model == 'bert':
        q_emb    = bert_model.encode([query[:MAX_CHARS]], normalize_embeddings=True, convert_to_numpy=True)
        clf_pred = le.inverse_transform(bert_svm.predict(q_emb))[0]
    elif model == 'nb':
        clf_pred = le.inverse_transform(nb_model.predict(tfidf.transform([query])))[0]
    else:
        clf_pred = le.inverse_transform(svm_model.predict(tfidf.transform([query])))[0]
    return {'predicted_label': clf_pred, 'weighted_label': best_label, 'predicted_amar': amars.get(best_label,''), 'top_k': top_k,
            'top_k_details': [{'case_id': cid, 'similarity': score, 'label': case_solutions.get(cid,{}).get('label',''), 'no_perkara': case_solutions.get(cid,{}).get('no_perkara',''), 'vonis': case_solutions.get(cid,{}).get('vonis','')} for cid, score in top_k]}

In [ ]:
DEMO = [
    {'id':'demo_001','true':'bersalah_ringan','query':'terdakwa bersama-sama melakukan kekerasan terang-terangan di muka umum menggunakan senjata tajam melukai korban'},
    {'id':'demo_002','true':'bersalah_ringan','query':'terdakwa memasuki rumah orang lain secara paksa tanpa izin menggunakan kekerasan mengancam penghuni'},
    {'id':'demo_003','true':'bersalah_berat','query':'terdakwa melakukan penghasutan kepada massa untuk kekerasan terhadap kelompok tertentu secara sistematis'},
    {'id':'demo_004','true':'bebas','query':'terdakwa tidak terbukti melakukan tindak pidana kekerasan dibebaskan dari segala dakwaan'},
    {'id':'demo_005','true':'bersalah_berat','query':'terdakwa bersama kelompok merusak fasilitas umum menyerang petugas keamanan saat demonstrasi'},
]
print('='*65)
print('DEMO PREDIKSI 5 KASUS BARU')
print('='*65)
demo_rows = []
for d in DEMO:
    r_svm  = predict_outcome(d['query'], model='svm')
    r_nb   = predict_outcome(d['query'], model='nb')
    r_bert = predict_outcome(d['query'], model='bert')
    ms = '✓' if r_svm['predicted_label']  == d['true'] else '✗'
    mn = '✓' if r_nb['predicted_label']   == d['true'] else '✗'
    mb = '✓' if r_bert['predicted_label'] == d['true'] else '✗'
    print(f"\n[{d['id']}] {d['query'][:70]}...")
    print(f"  True         : {d['true']}")
    print(f"  TF-IDF+SVM   : {r_svm['predicted_label']}  {ms}")
    print(f"  TF-IDF+NB    : {r_nb['predicted_label']}  {mn}")
    print(f"  IndoBERT+SVM : {r_bert['predicted_label']}  {mb}")
    demo_rows.append({'query_id': d['id'], 'query_text': d['query'], 'true_label': d['true'], 'pred_svm': r_svm['predicted_label'], 'pred_nb': r_nb['predicted_label'], 'pred_bert': r_bert['predicted_label'], 'top_5_ids': '|'.join([x[0] for x in r_svm['top_k']]), 'correct_svm': r_svm['predicted_label']==d['true'], 'correct_nb': r_nb['predicted_label']==d['true'], 'correct_bert': r_bert['predicted_label']==d['true']})
pd.DataFrame(demo_rows).to_csv(RESULTS_DIR / 'predictions_demo.csv', index=False, encoding='utf-8-sig')
print('\n✓ predictions_demo.csv disimpan')

---
## Tahap 5: Model Evaluation

In [ ]:
pred_svm  = svm_model.predict(X_test)
pred_nb   = nb_model.predict(X_test)
pred_bert = bert_svm.predict(test_embeddings)
true_lbl      = le.inverse_transform(y_test)
pred_svm_lbl  = le.inverse_transform(pred_svm)
pred_nb_lbl   = le.inverse_transform(pred_nb)
pred_bert_lbl = le.inverse_transform(pred_bert)

hit_tfidf  = sum(1 for i in test_idx if ids[i] in [x[0] for x in retrieve(texts[i], k=5)])
hit_bert   = sum(1 for i in test_idx if ids[i] in [x[0] for x in retrieve_bert(texts[i], k=5)])
topk_tfidf = round(hit_tfidf / len(test_idx), 4)
topk_bert  = round(hit_bert  / len(test_idx), 4)

def hitung(true, pred, name, topk):
    return {'model': name, 'accuracy': round(accuracy_score(true, pred), 4), 'precision': round(precision_score(true, pred, average='weighted', zero_division=0), 4), 'recall': round(recall_score(true, pred, average='weighted', zero_division=0), 4), 'f1_score': round(f1_score(true, pred, average='weighted', zero_division=0), 4), 'top_5_retrieval_accuracy': topk, 'n_test': len(true)}

metrics_df = pd.DataFrame([
    hitung(true_lbl, pred_svm_lbl,  'TF-IDF + SVM',         topk_tfidf),
    hitung(true_lbl, pred_nb_lbl,   'TF-IDF + Naive Bayes', topk_tfidf),
    hitung(true_lbl, pred_bert_lbl, 'IndoBERT + SVM',       topk_bert),
])
metrics_df.to_csv(EVAL_DIR / 'retrieval_metrics.csv', index=False, encoding='utf-8-sig')
print('='*60)
print('TABEL PERBANDINGAN 3 MODEL')
print('='*60)
display(metrics_df.set_index('model').style.background_gradient(subset=['accuracy','precision','recall','f1_score'], cmap='YlGn').format('{:.4f}', subset=['accuracy','precision','recall','f1_score','top_5_retrieval_accuracy']))

In [ ]:
for pred, name in zip([pred_svm_lbl, pred_nb_lbl, pred_bert_lbl], ['TF-IDF + SVM', 'TF-IDF + Naive Bayes', 'IndoBERT + SVM']):
    print(f'\nClassification Report – {name}:')
    print(classification_report(true_lbl, pred, zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col, title, color in zip(axes, ['accuracy','precision','recall','f1_score'], ['Accuracy','Precision','Recall','F1-Score'], ['#4C72B0','#DD8452','#55A868','#C44E52']):
    bars = ax.bar(metrics_df['model'], metrics_df[col], color=color, alpha=0.85, width=0.5)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylim(0, 1.2); ax.set_ylabel('Score'); ax.tick_params(axis='x', rotation=20)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2., h+0.02, f'{h:.4f}', ha='center', fontsize=8)
plt.suptitle('Perbandingan 3 Model CBR – Ketertiban Umum\nTF-IDF+SVM vs TF-IDF+NB vs IndoBERT+SVM', fontsize=11, y=1.05)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ figures/metrics_comparison.png disimpan')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
lbl_unik = sorted(set(true_lbl)|set(pred_svm_lbl)|set(pred_nb_lbl)|set(pred_bert_lbl))
for ax, pred, title, cmap in zip(axes, [pred_svm_lbl, pred_nb_lbl, pred_bert_lbl], ['TF-IDF + SVM','TF-IDF + Naive Bayes','IndoBERT + SVM'], ['Blues','Oranges','Greens']):
    cm = confusion_matrix(true_lbl, pred, labels=lbl_unik)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, xticklabels=lbl_unik, yticklabels=lbl_unik, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix\n{title}', fontweight='bold')
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ figures/confusion_matrix.png disimpan')

In [ ]:
errors = []
for i, ti in enumerate(test_idx):
    if true_lbl[i] != pred_svm_lbl[i]:
        row = df_labeled.iloc[ti]
        errors.append({'case_id': ids[ti], 'no_perkara': row['no_perkara'], 'true_label': true_lbl[i], 'pred_svm': pred_svm_lbl[i], 'pred_nb': pred_nb_lbl[i], 'pred_bert': pred_bert_lbl[i], 'word_count': row.get('word_count',0), 'vonis': row.get('vonis','')})
err_df = pd.DataFrame(errors)
err_df.to_csv(EVAL_DIR / 'error_analysis.csv', index=False, encoding='utf-8-sig')
print('ERROR ANALYSIS')
if len(err_df)==0:
    print('Tidak ada error pada test set!')
else:
    print(f'Total error: {len(err_df)}/{len(test_idx)} kasus')
    display(err_df)
    print('\nAnalisis penyebab error:')
    print('  - Test set kecil (4 kasus) → kurang representatif')
    print('  - 12 kasus berlabel tidak_diketahui tidak digunakan training')
    print('  - Rekomendasi: tambah data 50+ dokumen')
print('\n✓ error_analysis.csv disimpan')

In [ ]:
pred_rows = []
for i, ti in enumerate(test_idx):
    res = predict_outcome(texts[ti], model='svm')
    pred_rows.append({'query_id': f'q_{ids[ti]}', 'no_perkara': df_labeled.iloc[ti]['no_perkara'], 'true_label': true_lbl[i], 'predicted_label_svm': pred_svm_lbl[i], 'predicted_label_nb': pred_nb_lbl[i], 'predicted_label_bert': pred_bert_lbl[i], 'top_5_case_ids': '|'.join([x[0] for x in res['top_k']]), 'correct_svm': true_lbl[i]==pred_svm_lbl[i], 'correct_nb': true_lbl[i]==pred_nb_lbl[i], 'correct_bert': true_lbl[i]==pred_bert_lbl[i]})
pd.DataFrame(pred_rows).to_csv(EVAL_DIR / 'prediction_metrics.csv', index=False, encoding='utf-8-sig')
print('✓ prediction_metrics.csv disimpan')

---
## Kesimpulan & Rekomendasi

In [ ]:
best = metrics_df.loc[metrics_df['f1_score'].idxmax(), 'model']
print('='*60)
print('RINGKASAN HASIL')
print('='*60)
print(f'Total dokumen    : 31 putusan')
print(f'Kasus berlabel   : {len(df_labeled)}')
print(f'Split train/test : {len(train_idx)}/{len(test_idx)} (80:20)')
print()
for _, row in metrics_df.iterrows():
    print(f"{row['model']:25s} | Acc={row['accuracy']:.4f} | F1={row['f1_score']:.4f} | Top-5={row['top_5_retrieval_accuracy']:.4f}")
print(f'\nModel terbaik (F1): {best}')
print('\nRekomendasi Perbaikan:')
print('  1. Tambah data 50+ dokumen → hasil lebih representatif')
print('  2. Fine-tune IndoBERT dengan data putusan hukum Indonesia')
print('  3. Perbaiki ekstraksi label tidak_diketahui (12 kasus)')
print('  4. Gunakan k-fold cross validation untuk evaluasi lebih stabil')
print('  5. Ensemble TF-IDF + IndoBERT untuk retrieval hybrid')
print('\nOutput yang dihasilkan:')
for folder in [PROCESSED_DIR, EVAL_DIR, RESULTS_DIR, FIGS_DIR]:
    files = sorted(folder.glob('*'))
    if files:
        print(f'\n{folder.relative_to(ROOT_DIR)}/')
        for f in files: print(f'  ├── {f.name} ({f.stat().st_size/1024:.1f} KB)')